<a href="https://colab.research.google.com/github/martinpa-spec/Proyecto-Integrador-Henry-M5/blob/developer/preprocesamiento_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ Preprocesamiento y Preparación de Datos

En este notebook se ejecutan las transformaciones, depuraciones e imputaciones definidas a partir de las reglas de negocio identificadas en la fase de EDA (`comprension_eda.ipynb`).

In [2]:
import pandas as pd
import numpy as np

# Carga de la base de datos
df = pd.read_excel('Base_de_datos.xlsx')

# Unificación de tipos de datos básicos y nulos
df.replace(['N/A', '-', 'NaN', 'null', ''], np.nan, inplace=True)
df['tipo_credito'] = df['tipo_credito'].astype(str)
cols_moneda = ['salario_cliente', 'total_otros_prestamos', 'cuota_pactada']
df[cols_moneda] = df[cols_moneda].astype(float)

print("Forma del dataset original:", df.shape)

Forma del dataset original: (10763, 23)


### 1. DEPURACIÓN DE INCONSISTENCIAS Y OUTLIERS

In [3]:
# Regla 1: Eliminar registros con edad biológicamente imposible (> 100)
df = df[df['edad_cliente'] <= 100].copy()

# Regla 2: Convertir puntajes negativos a NaN para su posterior imputación
df.loc[df['puntaje'] < 0, 'puntaje'] = np.nan

# Regla 3: Control de salario extremo aberrante (> 30 Millones)
df = df[df['salario_cliente'] < 30000000].copy()

# Regla 4: Imputación de saldos nulos con 0.0 (Ausencia de deuda activa)
cols_saldos = ['saldo_mora', 'saldo_total', 'saldo_principal', 'saldo_mora_codeudor']
df[cols_saldos] = df[cols_saldos].fillna(0.0)

print("Forma del dataset tras la limpieza inicial:", df.shape)
print("Nulos restantes en saldos:", df[cols_saldos].isnull().sum().sum())

Forma del dataset tras la limpieza inicial: (10561, 23)
Nulos restantes en saldos: 0


### 2. IMPUTACIÓN DE VALORES FALTANTES (NULOS)

In [4]:
# 1. Variables cuantitativas de DataCrédito -> Imputación por Mediana
cols_num_imput = ['puntaje_datacredito', 'promedio_ingresos_datacredito', 'puntaje']

for col in cols_num_imput:
    mediana_val = df[col].median()
    df[col] = df[col].fillna(mediana_val)

# 2. Variables categóricas -> Imputación por Moda (Valor más frecuente)
cols_cat_imput = ['tendencia_ingresos']

for col in cols_cat_imput:
    moda_val = df[col].mode()[0]
    df[col] = df[col].fillna(moda_val)

# Verificación final: No deben quedar valores nulos en la base
print("--- Cantidad total de nulos tras la imputación ---")
print(df.isnull().sum().sum())

--- Cantidad total de nulos tras la imputación ---
0


###3. CODIFICACIÓN (ONE-HOT ENCODING)

In [5]:
# Variables categóricas a transformar
cols_categoricas = ['tipo_laboral', 'tipo_credito', 'tendencia_ingresos']

# Aplicar One-Hot Encoding (pd.get_dummies)
df_procesado = pd.get_dummies(df, columns=cols_categoricas, drop_first=True, dtype=int)

print("Forma del dataset tras One-Hot Encoding:", df_procesado.shape)
print("Nuevas columnas creadas:")
print([col for col in df_procesado.columns if any(c in col for c in cols_categoricas)])

Forma del dataset tras One-Hot Encoding: (10561, 70)
Nuevas columnas creadas:
['tipo_laboral_Independiente', 'tipo_credito_4', 'tipo_credito_6', 'tipo_credito_68', 'tipo_credito_7', 'tipo_credito_9', 'tendencia_ingresos_-566272', 'tendencia_ingresos_-435177', 'tendencia_ingresos_-224714', 'tendencia_ingresos_-164315', 'tendencia_ingresos_-101368', 'tendencia_ingresos_-70715', 'tendencia_ingresos_-28589', 'tendencia_ingresos_-4105', 'tendencia_ingresos_-288', 'tendencia_ingresos_0', 'tendencia_ingresos_3978', 'tendencia_ingresos_5697', 'tendencia_ingresos_8315', 'tendencia_ingresos_9090', 'tendencia_ingresos_9147', 'tendencia_ingresos_10808', 'tendencia_ingresos_15090', 'tendencia_ingresos_15245', 'tendencia_ingresos_17181', 'tendencia_ingresos_22363', 'tendencia_ingresos_22832', 'tendencia_ingresos_24702', 'tendencia_ingresos_31837', 'tendencia_ingresos_52862', 'tendencia_ingresos_54683', 'tendencia_ingresos_65988', 'tendencia_ingresos_75761', 'tendencia_ingresos_77975', 'tendencia_ing

### 4. GUARDAR DATASET PREPROCESADO

In [6]:
# Guardar en CSV listo para la fase de modelado
df_procesado.to_csv('base_datos_preprocesada.csv', index=False)

print("✅ Dataset exportado exitosamente como 'base_datos_preprocesada.csv'")

✅ Dataset exportado exitosamente como 'base_datos_preprocesada.csv'
